# 04 - Scouting: busqueda del refuerzo

Busqueda del central que corrige la carencia detectada en el notebook 03: **anticipar por
encima de la mediana sin perder juego aereo**.

Tres partes:

1. Un modelo supervisado que intenta predecir que centrales protegen su area. **No
   funciona**, y ese resultado negativo delimita lo que el trabajo puede afirmar.
2. Agrupacion no supervisada de los perfiles de central, y criba por criterios medibles
   fijados antes de mirar ningun nombre.
3. Perfil de los ocho finalistas y eleccion del refuerzo.

Al final queda documentada, como anexo, una via anterior que se descarto: un clasificador
de posicion aplicado a Segunda Division.

Lee `metricas_jugadores_laliga_2526.csv`, que genera el notebook 03.

## 1. El modelo supervisado: protegen el area esos registros?

La pregunta es si las cuatro metricas defensivas de un central permiten predecir si su
equipo concede poco o mucho xG en el centro del area en juego abierto.

Se etiqueta cada equipo segun quede por encima o por debajo de la mediana de la liga en
esa zona, se le pasa la etiqueta a sus centrales y se entrenan dos modelos: un arbol de
un nivel y un bosque aleatorio.

In [1]:
# === MODELO · ¿Qué acciones de un central protegen su área? ==============
from pathlib import Path
import pandas as pd
import numpy as np
from unidecode import unidecode

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
CARPETA_DATOS = RAIZ / "data"

d = pd.read_csv(CARPETA_DATOS / "disparos_laliga_2526_limpio.csv")
eqs = pd.read_csv(CARPETA_DATOS / "equipos_largo_laliga_2526.csv")
m = pd.read_csv(CARPETA_DATOS / "metricas_jugadores_laliga_2526.csv")

# --- 1. xG concedida en el centro del área, en juego abierto -------------
llave = eqs[["game", "equipo", "rival"]].drop_duplicates()
d2 = d.merge(llave, left_on=["game", "team"], right_on=["game", "equipo"], how="left")
d2 = d2.dropna(subset=["rival"])

X_AREA = 1 - 16.5 / 105
centro_area = d2[(d2["situation"] == "Open Play") &
                 (d2["location_x"] >= X_AREA) &
                 (d2["location_y"].between(1/3, 2/3))]

concedido = centro_area.groupby("rival")["xg"].sum()
UMBRAL_EQUIPO = concedido.median()
etiqueta_equipo = (concedido < UMBRAL_EQUIPO).astype(int)

print("xG CONCEDIDA EN EL CENTRO DEL ÁREA (juego abierto)")
print(f"Mediana de la liga: {UMBRAL_EQUIPO:.1f}")
print(concedido.sort_values().round(1).to_string())
print()
print(f"Mallorca: {concedido['Mallorca']:.1f}  ·  etiqueta {etiqueta_equipo['Mallorca']}")
print()

# --- 2. conjunto de entrenamiento: centrales con 600+ minutos ------------
VARIABLES = ["Interception_90", "Tackle_90", "Clearance_90", "Aerial_90"]
NOMBRES = ["Intercepciones", "Entradas", "Despejes", "Duelos aéreos"]

base = m[(m["posicion"] == "central") & (m["minutes"] >= 600)].dropna(subset=VARIABLES).copy()
base["y"] = base["equipo"].map(etiqueta_equipo)
base = base.dropna(subset=["y"])
base["y"] = base["y"].astype(int)

X = base[VARIABLES].values
y = base["y"].values
grupos = base["equipo"].values

print(f"Centrales en la muestra: {len(base)}  ·  equipos: {base['equipo'].nunique()}")
print(f"Reparto de clases: {np.bincount(y)}  (0 = concede mucho, 1 = concede poco)")
print()

# --- 3. validación particionando por equipo ------------------------------
# los jugadores de un mismo club comparten etiqueta: si se reparten entre
# entrenamiento y evaluación, el modelo puede acertar reconociendo al equipo
particion = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Árbol de un nivel": DecisionTreeClassifier(max_depth=1, random_state=42),
    "Bosque aleatorio": RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=3, random_state=42),
}

for nombre, modelo in modelos.items():
    pred = cross_val_predict(modelo, X, y, cv=particion, groups=grupos)
    print(f"--- {nombre}")
    print(f"Acierto: {accuracy_score(y, pred):.3f}   "
          f"(clase mayoritaria: {max(np.bincount(y)) / len(y):.3f})")
    print("Matriz de confusión (filas: real, columnas: predicho)")
    print(confusion_matrix(y, pred))
    print(classification_report(y, pred, target_names=["concede mucho", "concede poco"],
                                zero_division=0))

# --- 4. qué variable usa cada modelo -------------------------------------
arbol = DecisionTreeClassifier(max_depth=1, random_state=42).fit(X, y)
i = arbol.tree_.feature[0]
print(f"UMBRAL APRENDIDO POR EL ÁRBOL: {NOMBRES[i]} <= {arbol.tree_.threshold[0]:.3f}")
print()

bosque = RandomForestClassifier(n_estimators=300, max_depth=4,
                                min_samples_leaf=3, random_state=42).fit(X, y)
print("IMPORTANCIA DE CADA VARIABLE (bosque aleatorio)")
for n, imp in sorted(zip(NOMBRES, bosque.feature_importances_),
                     key=lambda t: -t[1]):
    print(f"  {n:<16} {imp:.3f}")
print()

# --- 5. puntuación de los candidatos -------------------------------------
CANDIDATOS = ["diego llorente", "zaid romero", "luiz felipe", "andres castrin",
              "eray comert", "alejandro frances", "unai nunez", "david carmo"]
MALLORCA = ["martin valjent", "raillo", "david lopez", "marash kumbulla"]

ficha = m[(m["posicion"] == "central")].dropna(subset=VARIABLES).copy()
ficha["prob"] = bosque.predict_proba(ficha[VARIABLES].values)[:, 1]

print("PROBABILIDAD DE PERFIL QUE PROTEGE EL ÁREA")
print("\nCandidatos:")
print(ficha[ficha["clave"].isin(CANDIDATOS)][["clave", "equipo", "minutes", "prob"]]
      .sort_values("prob", ascending=False).round(3).to_string(index=False))
print("\nCentrales del Mallorca:")
print(ficha[ficha["clave"].isin(MALLORCA)][["clave", "minutes", "prob"]]
      .sort_values("prob", ascending=False).round(3).to_string(index=False))

xG CONCEDIDA EN EL CENTRO DEL ÁREA (juego abierto)
Mediana de la liga: 31.1
rival
Real Madrid        25.2
Real Betis         27.0
Getafe             27.6
Osasuna            27.7
Atletico Madrid    29.0
Valencia           29.1
Villarreal         29.1
Celta Vigo         30.3
Rayo Vallecano     30.8
Alaves             31.0
Athletic Club      31.2
Barcelona          33.0
Real Sociedad      36.9
Mallorca           38.6
Sevilla            38.7
Espanyol           39.5
Girona             40.2
Real Oviedo        41.0
Elche              44.0
Levante            44.8

Mallorca: 38.6  ·  etiqueta 0

Centrales en la muestra: 65  ·  equipos: 20
Reparto de clases: [35 30]  (0 = concede mucho, 1 = concede poco)

--- Árbol de un nivel
Acierto: 0.523   (clase mayoritaria: 0.538)
Matriz de confusión (filas: real, columnas: predicho)
[[ 6 29]
 [ 2 28]]
               precision    recall  f1-score   support

concede mucho       0.75      0.17      0.28        35
 concede poco       0.49      0.93      0.64 

--- Bosque aleatorio
Acierto: 0.492   (clase mayoritaria: 0.538)
Matriz de confusión (filas: real, columnas: predicho)
[[19 16]
 [17 13]]
               precision    recall  f1-score   support

concede mucho       0.53      0.54      0.54        35
 concede poco       0.45      0.43      0.44        30

     accuracy                           0.49        65
    macro avg       0.49      0.49      0.49        65
 weighted avg       0.49      0.49      0.49        65

UMBRAL APRENDIDO POR EL ÁRBOL: Duelos aéreos <= 4.164



IMPORTANCIA DE CADA VARIABLE (bosque aleatorio)
  Duelos aéreos    0.307
  Despejes         0.275
  Intercepciones   0.245
  Entradas         0.174

PROBABILIDAD DE PERFIL QUE PROTEGE EL ÁREA

Candidatos:
            clave         equipo  minutes  prob
      eray comert       Valencia     1341 0.722
      zaid romero         Getafe     1350 0.635
       unai nunez       Valencia      927 0.607
   diego llorente     Real Betis     1221 0.567
   andres castrin        Sevilla     1251 0.539
alejandro frances         Girona     1077 0.411
      luiz felipe Rayo Vallecano      473 0.332
      david carmo    Real Oviedo     1785 0.269

Centrales del Mallorca:
          clave  minutes  prob
 martin valjent     3311 0.507
marash kumbulla      434 0.359
    david lopez      300 0.286
    david lopez      827 0.258
         raillo     2069 0.103


**Ningun modelo supera al clasificador trivial.** El arbol de un nivel acierta el 52,3 %
y el bosque aleatorio el 49,2 %, frente al 53,8 % que se consigue prediciendo siempre la
clase mayoritaria.

La validacion particiona **por equipo** con `StratifiedGroupKFold`: los centrales de un
mismo club comparten etiqueta, y si se reparten entre entrenamiento y evaluacion el modelo
puede acertar reconociendo al equipo en lugar de aprender nada util. Una version anterior
de este analisis no lo hacia y daba resultados inflados.

Que el modelo no prediga es un resultado, y es el que marca el limite del trabajo: los
registros defensivos individuales de un central no permiten anticipar cuanto xG concede su
equipo en el centro del area. El analisis puede identificar el perfil que falta y senalar
quien lo acredita, pero no cuantificar cuantos goles esperados se evitarian con el.

## 2. Comprobacion del fichero de metricas

Dos comprobaciones sobre el fichero maestro antes de seguir: que no hay filas repetidas
por jugador y equipo, y cual es el unico nombre que aparece dos veces.

In [2]:
# === Comprobación de duplicados en el fichero de métricas ================
m = pd.read_csv(CARPETA_DATOS / "metricas_jugadores_laliga_2526.csv")

dup = m[m.duplicated(subset=["clave", "equipo"], keep=False)]
print(f"Filas duplicadas por jugador y equipo: {len(dup)}")
if len(dup):
    print(dup[["clave", "equipo", "posicion", "minutes"]]
          .sort_values(["clave", "minutes"]).to_string(index=False))
print()
print(f"Jugadores únicos: {m['clave'].nunique()}   Filas: {len(m)}")

Filas duplicadas por jugador y equipo: 0

Jugadores únicos: 480   Filas: 481


In [3]:
# === Qué nombre está repetido y en qué equipos ===========================
repes = m[m.duplicated(subset=["clave"], keep=False)]
print(repes[["clave", "player", "equipo", "team", "posicion", "minutes"]]
      .sort_values(["clave", "minutes"]).to_string(index=False))

      clave      player   equipo     team posicion  minutes
david lopez David López   Girona   Girona  central      300
david lopez David López Mallorca Mallorca  central      827


481 filas y 480 nombres unicos. El unico repetido es **David Lopez**: uno en el Mallorca
(827 minutos) y otro en el Girona (300). Por eso todos los cruces se hacen por equipo y
jugador, nunca solo por nombre.

## 3. Perfiles de central: agrupacion no supervisada

Si el modelo supervisado no sirve para predecir, al menos se puede describir. Un k-medias
sobre las cuatro metricas defensivas estandarizadas agrupa a los 65 centrales de la liga
por perfil, sin usar ninguna etiqueta de rendimiento del equipo.

In [4]:
# === MODELO NO SUPERVISADO · Perfiles de central =========================
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
CARPETA_DATOS = RAIZ / "data"

m = pd.read_csv(CARPETA_DATOS / "metricas_jugadores_laliga_2526.csv")

VARIABLES = ["Interception_90", "Tackle_90", "Clearance_90", "Aerial_90"]
NOMBRES = ["Intercepciones", "Entradas", "Despejes", "Duelos aéreos"]

# --- los grupos se definen sobre centrales con muestra sólida ------------
base = m[(m["posicion"] == "central") & (m["minutes"] >= 600)].dropna(subset=VARIABLES).copy()

escala = StandardScaler()
X = escala.fit_transform(base[VARIABLES].values)

print(f"Centrales usados para definir los perfiles: {len(base)}")
print()
print("CALIDAD DE LA AGRUPACIÓN SEGÚN EL NÚMERO DE GRUPOS")
for k in range(2, 6):
    modelo = KMeans(n_clusters=k, n_init=25, random_state=42).fit(X)
    print(f"  k = {k}   silueta = {silhouette_score(X, modelo.labels_):.3f}")
print()

K = 3   # cámbialo si la silueta señala otro valor
km = KMeans(n_clusters=K, n_init=25, random_state=42).fit(X)
base["grupo"] = km.labels_

# --- cómo es cada grupo, en unidades reales ------------------------------
centros = pd.DataFrame(escala.inverse_transform(km.cluster_centers_), columns=NOMBRES)
centros.index.name = "grupo"
centros["jugadores"] = base["grupo"].value_counts().sort_index().values
print("PERFIL MEDIO DE CADA GRUPO")
print(centros.round(2).to_string())
print()

GRUPO_ANTICIPA = int(centros["Intercepciones"].idxmax())
print(f"Grupo con más anticipación: {GRUPO_ANTICIPA}")
print()

# --- dónde caen los centrales del Mallorca -------------------------------
mall = base[base["equipo"].str.lower() == "mallorca"]
print("CENTRALES DEL MALLORCA")
print(mall[["clave", "minutes", "grupo"] + VARIABLES].round(2).to_string(index=False))
print()

# Kumbulla no entra en la muestra de 600 min: se le asigna grupo aparte
kum = m[(m["posicion"] == "central") &
        (m["equipo"].str.lower() == "mallorca") &
        (m["minutes"] < 600)].dropna(subset=VARIABLES)
if len(kum):
    g = km.predict(escala.transform(kum[VARIABLES].values))
    kum = kum.assign(grupo=g)
    print("CENTRAL DEL MALLORCA FUERA DE LA MUESTRA (asignado al grupo más cercano)")
    print(kum[["clave", "minutes", "grupo"] + VARIABLES].round(2).to_string(index=False))
    print()

# --- candidatos: fondo de rotación, dentro del grupo de anticipación -----
fondo = m[(m["posicion"] == "central") &
          (m["minutes"].between(400, 1800))].dropna(subset=VARIABLES).copy()
fondo["grupo"] = km.predict(escala.transform(fondo[VARIABLES].values))

sel = fondo[fondo["grupo"] == GRUPO_ANTICIPA].sort_values("Interception_90", ascending=False)
print(f"CANDIDATOS DEL GRUPO {GRUPO_ANTICIPA} CON 400-1800 MINUTOS: {len(sel)}")
print(sel[["clave", "equipo", "minutes"] + VARIABLES].round(2).to_string(index=False))
print()

print("REPARTO DE TODO EL FONDO POR GRUPO")
print(fondo["grupo"].value_counts().sort_index().to_string())

Centrales usados para definir los perfiles: 65

CALIDAD DE LA AGRUPACIÓN SEGÚN EL NÚMERO DE GRUPOS


  k = 2   silueta = 0.234
  k = 3   silueta = 0.201
  k = 4   silueta = 0.210


  k = 5   silueta = 0.221

PERFIL MEDIO DE CADA GRUPO
       Intercepciones  Entradas  Despejes  Duelos aéreos  jugadores
grupo                                                              
0                0.88      1.39      6.54           4.27         25
1                1.32      1.57      5.78           3.16         23
2                0.95      2.10      4.22           2.52         17

Grupo con más anticipación: 1

CENTRALES DEL MALLORCA
         clave  minutes  grupo  Interception_90  Tackle_90  Clearance_90  Aerial_90
        raillo     2069      0             0.70       1.09          5.31       4.96
martin valjent     3311      0             0.63       1.36          6.74       2.66
   david lopez      827      0             0.87       1.63         10.01       5.66

CENTRAL DEL MALLORCA FUERA DE LA MUESTRA (asignado al grupo más cercano)
          clave  minutes  grupo  Interception_90  Tackle_90  Clearance_90  Aerial_90
marash kumbulla      434      0             0.41       0

Con tres grupos salen tres perfiles claros: uno que despeja y gana duelos (0,88
intercepciones de media), uno que anticipa (1,32) y uno que entra al corte (2,10 entradas
con muchos menos despejes).

**Los cuatro centrales del Mallorca caen todos en el grupo que menos anticipa**, Kumbulla
incluido. No es un jugador concreto, es el perfil de toda la posicion.

Aplicando los grupos al fondo de rotacion (400 a 1.800 minutos) salen 14 candidatos del
grupo que anticipa, con Diego Llorente y Zaid Romero en cabeza.

*La silueta es baja para todos los valores de k, entre 0,20 y 0,23, asi que los grupos no
estan bien separados. Sirven para describir perfiles, no como clasificacion solida.*

## 4. Comprobacion con dos grupos

La silueta es algo mejor con dos grupos que con tres, asi que se repite la agrupacion para
comprobar que la conclusion no depende de esa eleccion.

In [5]:
# === Comparación con dos grupos, que es lo que indica la silueta =========
km2 = KMeans(n_clusters=2, n_init=25, random_state=42).fit(X)
base["grupo2"] = km2.labels_

centros2 = pd.DataFrame(escala.inverse_transform(km2.cluster_centers_), columns=NOMBRES)
centros2.index.name = "grupo"
centros2["jugadores"] = base["grupo2"].value_counts().sort_index().values
print("PERFIL MEDIO CON DOS GRUPOS")
print(centros2.round(2).to_string())
print()

G2 = int(centros2["Intercepciones"].idxmax())
print(f"Grupo con más anticipación: {G2}")
print()

print("CENTRALES DEL MALLORCA")
print(base[base["equipo"].str.lower() == "mallorca"][["clave", "minutes", "grupo", "grupo2"]]
      .to_string(index=False))

kum = m[(m["posicion"] == "central") & (m["equipo"].str.lower() == "mallorca") &
        (m["minutes"] < 600)].dropna(subset=VARIABLES)
if len(kum):
    print(f"kumbulla -> grupo2 = {km2.predict(escala.transform(kum[VARIABLES].values))[0]}")
print()

fondo = m[(m["posicion"] == "central") &
          (m["minutes"].between(400, 1800))].dropna(subset=VARIABLES).copy()
fondo["grupo2"] = km2.predict(escala.transform(fondo[VARIABLES].values))
sel2 = fondo[fondo["grupo2"] == G2].sort_values("Interception_90", ascending=False)

print(f"CANDIDATOS CON DOS GRUPOS: {len(sel2)}")
print(sel2[["clave", "equipo", "minutes"] + VARIABLES].round(2).to_string(index=False))

PERFIL MEDIO CON DOS GRUPOS
       Intercepciones  Entradas  Despejes  Duelos aéreos  jugadores
grupo                                                              
0                1.21      1.94      4.96           2.83         31
1                0.91      1.36      6.31           3.96         34

Grupo con más anticipación: 0

CENTRALES DEL MALLORCA
         clave  minutes  grupo  grupo2
        raillo     2069      0       1
martin valjent     3311      0       1
   david lopez      827      0       1
kumbulla -> grupo2 = 1

CANDIDATOS CON DOS GRUPOS: 15
            clave          equipo  minutes  Interception_90  Tackle_90  Clearance_90  Aerial_90
    yeray alvarez   Athletic Club      517             2.26       0.17          5.74       3.31
   diego llorente      Real Betis     1221             2.06       1.99          7.37       4.05
      zaid romero          Getafe     1350             1.87       2.07          6.33       3.47
    aitor paredes   Athletic Club     1625         

Con dos grupos el resultado no cambia: los cuatro centrales del Mallorca siguen juntos en
el grupo que menos anticipa, y la lista de candidatos pasa de 14 a 15 con los mismos
nombres en cabeza. La conclusion no depende del numero de grupos.

## 5. La criba por criterios

Los criterios se fijan antes de mirar ningun nombre, para no elegir primero al jugador y
buscarle despues los numeros:

1. **Posicion**: central.
2. **Minutos**: entre 400 y 1.800, que es el rol de rotacion y por tanto lo accesible para
   el presupuesto del Mallorca.
3. **Intercepciones por 90**: al menos 1,06, la mediana de los 65 centrales.
4. **Entradas por 90**: al menos 1,58, la mediana de los 65 centrales.

Los dos umbrales de rendimiento son medianas, no valores elegidos a mano: el refuerzo
tiene que anticipar por encima de lo normal en su puesto.

Antes se revisa el caso de Marash Kumbulla, el cuarto central de la plantilla, por si el
recambio estaba en casa.

In [6]:
# === 3.5 · Criba con suelo de 400 minutos + caso Kumbulla ================
from pathlib import Path
import pandas as pd
from unidecode import unidecode

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
CARPETA_DATOS = RAIZ / "data"

m = pd.read_csv(CARPETA_DATOS / "metricas_jugadores_laliga_2526.csv")
ali = pd.read_csv(CARPETA_DATOS / "alineaciones_laliga_2526.csv")
alt = pd.read_csv(CARPETA_DATOS / "altura_defensiva_laliga_2526.csv")

DEF = ["Interception_90", "Tackle_90", "Clearance_90", "Aerial_90"]
MIN_SUELO, MIN_TECHO = 400, 1800

# --- grupos de referencia: se quedan en 600 min, como el capítulo 3 -------
ref_liga = m[(m["posicion"] == "central") & (m["minutes"] >= 600)].dropna(subset=DEF)
BLOQUE_BAJO = alt.sort_values("altura_defensiva")["equipo"].head(6).tolist()
ref_bajo = ref_liga[ref_liga["equipo"].isin(BLOQUE_BAJO)]

UMBRAL_INT = ref_liga["Interception_90"].median()
UMBRAL_TKL = ref_liga["Tackle_90"].median()

print(f"Referencia LaLiga: {len(ref_liga)} centrales (>= 600 min)")
print(f"Referencia bloque bajo: {len(ref_bajo)} centrales  {BLOQUE_BAJO}")
print(f"Umbrales: intercepciones {UMBRAL_INT:.2f}  ·  entradas {UMBRAL_TKL:.2f}")
print()

# --- edad --------------------------------------------------------------
def normalizar(nombre):
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    return " ".join(n.split())

edades = ali.groupby("player")["edad"].max().reset_index()
edades["clave_ali"] = edades["player"].apply(normalizar)

# --- caso Kumbulla ------------------------------------------------------
kum = m[(m["posicion"] == "central") &
        (m["equipo"].str.lower() == "mallorca") &
        (m["clave"].str.contains("kumbulla", case=False, na=False))]

print("MARASH KUMBULLA")
if len(kum) == 0 or kum[DEF].isna().all(axis=None):
    print("Sin métricas cruzadas: no se puede analizar.")
else:
    fila = kum.iloc[0]
    print(f"{int(fila['minutes'])} minutos")
    for met in DEF:
        p_liga = round((ref_liga[met].dropna() < fila[met]).mean() * 100)
        p_bajo = round((ref_bajo[met].dropna() < fila[met]).mean() * 100)
        print(f"  {met.replace('_90',''):<14} {fila[met]:5.2f}   "
              f"p{p_liga} liga  ·  p{p_bajo} bloque bajo   "
              f"(medianas {ref_liga[met].median():.2f} / {ref_bajo[met].median():.2f})")
print()

# --- fondo de búsqueda con suelo de 400 ---------------------------------
centrales400 = m[(m["posicion"] == "central") & (m["minutes"] >= MIN_SUELO)]
sin_datos = centrales400[centrales400["Interception_90"].isna()]
if len(sin_datos):
    print("CENTRALES SIN MÉTRICAS CRUZADAS (excluidos)")
    print(sin_datos[["clave", "equipo", "minutes"]].to_string(index=False))
    print()

base = centrales400.dropna(subset=DEF).merge(
    edades[["clave_ali", "edad"]], left_on="clave", right_on="clave_ali", how="left")

fondo = base[(base["minutes"] >= MIN_SUELO) & (base["minutes"] <= MIN_TECHO)]
print(f"FONDO DE BÚSQUEDA: {len(fondo)} centrales ({MIN_SUELO}-{MIN_TECHO} min)")
print()

criba = fondo[(fondo["Interception_90"] >= UMBRAL_INT) &
              (fondo["Tackle_90"] >= UMBRAL_TKL)].sort_values(
    "Interception_90", ascending=False)

cols = ["clave", "equipo", "edad", "minutes"] + DEF
print(f"FINALISTAS: {len(criba)}")
print(criba[cols].round(2).to_string(index=False))
print()
print(f"Edad sin cruzar: {criba['edad'].isna().sum()} de {len(criba)}")
print()

# --- los que se quedan a las puertas, por si interesa ampliar ------------
cerca = fondo[((fondo["Interception_90"] >= UMBRAL_INT) &
               (fondo["Tackle_90"] < UMBRAL_TKL)) |
              ((fondo["Interception_90"] < UMBRAL_INT) &
               (fondo["Interception_90"] >= UMBRAL_INT * 0.9) &
               (fondo["Tackle_90"] >= UMBRAL_TKL))].sort_values(
    "Interception_90", ascending=False)
print(f"SE QUEDAN CERCA: {len(cerca)}")
print(cerca[cols].round(2).to_string(index=False))

Referencia LaLiga: 65 centrales (>= 600 min)
Referencia bloque bajo: 21 centrales  ['Mallorca', 'Villarreal', 'Girona', 'Real Oviedo', 'Osasuna', 'Celta Vigo']
Umbrales: intercepciones 1.06  ·  entradas 1.58

MARASH KUMBULLA
434 minutos
  Interception    0.41   p3 liga  ·  p0 bloque bajo   (medianas 1.06 / 0.96)
  Tackle          0.62   p0 liga  ·  p0 bloque bajo   (medianas 1.58 / 1.63)
  Clearance       2.90   p6 liga  ·  p5 bloque bajo   (medianas 5.78 / 6.09)
  Aerial          3.32   p43 liga  ·  p57 bloque bajo   (medianas 3.36 / 3.02)

CENTRALES SIN MÉTRICAS CRUZADAS (excluidos)


                clave equipo  minutes
jon pacheco dozagarat Alaves     1737

FONDO DE BÚSQUEDA: 33 centrales (400-1800 min)

FINALISTAS: 8
            clave         equipo  edad  minutes  Interception_90  Tackle_90  Clearance_90  Aerial_90
   diego llorente     Real Betis  33.0     1221             2.06       1.99          7.37       4.05
      zaid romero         Getafe  26.0     1350             1.87       2.07          6.33       3.47
      luiz felipe Rayo Vallecano  29.0      473             1.52       2.28          4.57       3.23
   andres castrin        Sevilla  23.0     1251             1.44       1.87          6.12       3.45
      eray comert       Valencia  28.0     1341             1.34       1.74          5.91       2.08
alejandro frances         Girona  24.0     1077             1.25       2.09          5.10       1.67
       unai nunez       Valencia  29.0      927             1.07       2.43          4.37       2.43
      david carmo    Real Oviedo  27.0     1785      

**De 65 centrales, 33 cumplen el rango de minutos y 8 superan los cuatro criterios.**

Kumbulla queda descartado como recambio interno: percentil 3 de la liga en intercepciones
y percentil 0 contra el grupo de bloque bajo.

Un central se queda fuera por falta de datos y conviene declararlo: **Jon Pacheco**
(Alaves, 1.737 minutos) no tiene ninguna de las cuatro metricas cruzadas. Esa es la
diferencia entre los 66 centrales con 600 minutos o mas y los 65 con las cuatro metricas
completas que usa el trabajo.

## 6. Perfil de los ocho finalistas

De cada finalista se miran los percentiles dentro de los 65 centrales de la liga, la altura
defensiva de su equipo (para saber en que contexto registra esos numeros) y dos metricas
de salida de balon.

In [7]:
# === 4 · Perfil de los ocho finalistas ===================================
FINALISTAS = [
    "diego llorente", "zaid romero", "luiz felipe", "andres castrin",
    "eray comert", "alejandro frances", "unai nunez", "david carmo",
]

alt_dic = dict(zip(alt["equipo"], alt["altura_defensiva"]))
ALTURA_MALLORCA = alt_dic["Mallorca"]

SALIDA = [c for c in ["xg_buildup_90", "key_passes_90"] if c in m.columns]

print(f"Altura defensiva del Mallorca: {ALTURA_MALLORCA:.2f} m")
print(f"Mediana de LaLiga: {alt['altura_defensiva'].median():.2f} m")
print()

for nombre in FINALISTAS:
    fila = criba[criba["clave"] == nombre]
    if len(fila) == 0:
        print(f"{nombre}: no encontrado en la criba")
        continue
    fila = fila.iloc[0]

    altura = alt_dic.get(fila["equipo"], float("nan"))
    aviso = "  [MUESTRA CORTA]" if fila["minutes"] < 600 else ""

    print(f"{fila['clave'].upper()}  ·  {fila['equipo']}  ·  "
          f"{int(fila['edad'])} años  ·  {int(fila['minutes'])} min{aviso}")
    print(f"  Altura defensiva del equipo: {altura:.2f} m "
          f"({altura - ALTURA_MALLORCA:+.2f} m respecto al Mallorca)")

    for met in DEF:
        p = round((ref_liga[met].dropna() < fila[met]).mean() * 100)
        print(f"  {met.replace('_90',''):<14} {fila[met]:5.2f}  (p{p})")

    for met in SALIDA:
        p = round((ref_liga[met].dropna() < fila[met]).mean() * 100)
        print(f"  {met.replace('_90',''):<14} {fila[met]:5.2f}  (p{p})")
    print()

# --- ¿intercepta más quien defiende más alto? ----------------------------
print("CONTROL · relación entre altura defensiva del equipo e intercepciones")
tmp = ref_liga.copy()
tmp["altura"] = tmp["equipo"].map(alt_dic)
print(f"Correlación (65 centrales): "
      f"{tmp['altura'].corr(tmp['Interception_90']):.3f}")

Altura defensiva del Mallorca: 29.80 m
Mediana de LaLiga: 32.24 m

DIEGO LLORENTE  ·  Real Betis  ·  33 años  ·  1221 min
  Altura defensiva del equipo: 32.05 m (+2.25 m respecto al Mallorca)
  Interception    2.06  (p98)
  Tackle          1.99  (p75)
  Clearance       7.37  (p89)
  Aerial          4.05  (p78)
  xg_buildup      0.28  (p63)
  key_passes      0.07  (p11)

ZAID ROMERO  ·  Getafe  ·  26 años  ·  1350 min
  Altura defensiva del equipo: 32.85 m (+3.05 m respecto al Mallorca)
  Interception    1.87  (p97)
  Tackle          2.07  (p82)
  Clearance       6.33  (p65)
  Aerial          3.47  (p62)
  xg_buildup      0.17  (p22)
  key_passes      0.13  (p26)

LUIZ FELIPE  ·  Rayo Vallecano  ·  29 años  ·  473 min  [MUESTRA CORTA]
  Altura defensiva del equipo: 32.26 m (+2.46 m respecto al Mallorca)
  Interception    1.52  (p91)
  Tackle          2.28  (p91)
  Clearance       4.57  (p20)
  Aerial          3.23  (p42)
  xg_buildup      0.15  (p15)
  key_passes      0.19  (p40)

ANDRE

**Diego Llorente destaca justo en lo que falta**: 2,06 intercepciones por 90, percentil 98
de la liga, el triple que Valjent, manteniendo 4,07 duelos aereos (percentil 78) y 7,37
despejes (percentil 89). Aporta anticipacion sin renunciar a la solidez aerea, que es
exactamente lo que pedia el diagnostico.

Andres Castrin es el otro finalista serio: 23 anos, percentil 86 en intercepciones y mucho
mas barato. Como operacion de mercado es mejor; por rendimiento inmediato, no.

La ultima comprobacion cierra la objecion tactica: **la correlacion entre la altura
defensiva del equipo y las intercepciones de sus centrales es de 0,108**, practicamente
nula. Los numeros de Llorente no se explican porque el Betis defienda mas alto.

*Limitacion a declarar: Luiz Felipe entra en la lista con 473 minutos, muestra corta para
fiarse de sus valores por 90.*

---
## Conclusiones del scouting

1. **El modelo supervisado no predice**: 0,523 y 0,492 de acierto frente al 0,538 del
   clasificador trivial. El trabajo identifica el perfil que falta, no cuantifica el efecto
   de ficharlo.
2. **Los cuatro centrales del Mallorca comparten perfil**: el grupo que menos anticipa,
   tanto con dos grupos como con tres. No habia recambio interno.
3. **La criba deja ocho candidatos** de los 65 centrales de LaLiga: posicion, 400 a 1.800
   minutos y por encima de la mediana en intercepciones y en entradas.
4. **El refuerzo propuesto es Diego Llorente**: percentil 98 en intercepciones sin perder
   juego aereo. La alternativa era Andres Castrin, mejor operacion de mercado pero con
   menos rendimiento inmediato.

El notebook 06 acota hasta donde llega esa propuesta con una simulacion de la temporada.

---
# Anexo. Via descartada: clasificador de posicion y Segunda Division

Lo que sigue es el primer enfoque del scouting, que se abandono y se deja documentado.

La idea era entrenar un clasificador que distinguiera central de lateral a partir de las
estadisticas de FBref, aplicarlo a Segunda Division y buscar alli el refuerzo. Llegaba a un
87,4 % de acierto y terminaba proponiendo a **Xavi Sintes** (Cordoba CF).

**Por que se descarto:**

- WhoScored no publica eventos de Segunda Division, asi que esos candidatos solo tenian
  intercepciones, entradas, faltas y centros de FBref. Sin despejes, sin bloqueos y sin
  duelos aereos no se puede comprobar que el refuerzo no empeora lo que ya funcionaba.
- No hay forma de ajustar por nivel competitivo. Interceptar en Segunda es mas facil porque
  el rival ejecuta peor, y las medianas de las dos categorias no son comparables aunque
  coincidan.
- El objetivo del clasificador era resolver la falta de posiciones en las fuentes, y eso
  quedo resuelto mejor con el fichero propio del notebook 03, validado al 92,0 % en
  centrales.

**Las cifras de este anexo son de esa version anterior del analisis y no coinciden con las
del trabajo entregado.** Se conservan porque documentan el proceso, no como resultado.

### A1. Carga de datos

In [8]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Importamos pandas para manejar los datos
import pandas as pd
# Importamos numpy para calculos numericos
import numpy as np
# Importamos unidecode para normalizar nombres
from unidecode import unidecode

# Definimos la funcion que normaliza un nombre
def normalizar(nombre):
    # Quitamos acentos, pasamos a minusculas y limpiamos separadores
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    # Eliminamos espacios sobrantes
    return " ".join(n.split())

# Cargamos la tabla de jugadores de LaLiga con posiciones y eventos
jug = pd.read_csv("data/jugadores_eventos_laliga_2526.csv")

# Nos quedamos con centrales y laterales, que son las clases a distinguir
defensas = jug[jug["posicion"].isin(["central", "lateral"])].copy()

# Calculamos las metricas por 90 minutos
defensas["crs_90"] = defensas["Crs"] / defensas["90s"]
defensas["fls_90"] = defensas["Fls"] / defensas["90s"]
defensas["crdy_90"] = defensas["CrdY"] / defensas["90s"]

# Definimos las variables candidatas para el modelo
variables = ["crs_90", "int_90", "tklw_90", "fls_90", "crdy_90"]

# Comparamos la mediana de cada variable entre centrales y laterales
print("MEDIANA POR POSICION")
print(defensas.groupby("posicion")[variables].median().round(2))

# Mostramos cuantos jugadores hay de cada clase
print("\nJugadores por clase:")
print(defensas["posicion"].value_counts())

MEDIANA POR POSICION
          crs_90  int_90  tklw_90  fls_90  crdy_90
posicion                                          
central      0.1    1.10     0.92    1.05     0.23
lateral      2.4    1.07     1.17    1.11     0.20

Jugadores por clase:
posicion
central    64
lateral    63
Name: count, dtype: int64


**Los centros al area separan las dos clases de forma clara**: 0,10 por 90 en centrales
frente a 2,40 en laterales, 24 veces mas. Las demas variables apenas distinguen
(intercepciones 1,10 frente a 1,07; faltas 1,05 frente a 1,11).

Con una diferencia asi conviene probar primero lo simple antes de montar un modelo complejo.

### A2. Comparacion de modelos

Los tres se evaluan con la **misma validacion cruzada** (5 particiones estratificadas),
prediciendo siempre sobre jugadores que el modelo no ha visto. Es la unica comparacion
honesta: evaluar un umbral sobre los mismos datos con los que se eligio infla el resultado.

In [9]:
# Importamos las herramientas de modelado
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Creamos la variable objetivo: 1 si es central, 0 si es lateral
defensas["es_central"] = (defensas["posicion"] == "central").astype(int)

# Preparamos las variables predictoras y la objetivo
X = defensas[variables]
y = defensas["es_central"]

# Definimos la validacion cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Creamos un arbol de un solo nivel, que equivale a un umbral aprendido
tocon = DecisionTreeClassifier(max_depth=1, random_state=42)

# Validamos el umbral usando solo los centros al area
p1 = cross_val_score(tocon, defensas[["crs_90"]], y, cv=cv, scoring="accuracy")
print(f"Umbral aprendido (solo centros) -> acierto: {round(p1.mean()*100, 1)}% (+/- {round(p1.std()*100, 1)})")

# Validamos una regresion logistica con las cinco variables escaladas
logistica = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
p2 = cross_val_score(logistica, X, y, cv=cv, scoring="accuracy")
print(f"Regresion logistica (5 vars)    -> acierto: {round(p2.mean()*100, 1)}% (+/- {round(p2.std()*100, 1)})")

# Validamos un bosque aleatorio con las cinco variables
bosque = RandomForestClassifier(n_estimators=300, random_state=42)
p3 = cross_val_score(bosque, X, y, cv=cv, scoring="accuracy")
print(f"Bosque aleatorio (5 vars)       -> acierto: {round(p3.mean()*100, 1)}% (+/- {round(p3.std()*100, 1)})")

# Entrenamos el bosque para ver la importancia de cada variable
bosque.fit(X, y)
print("\nImportancia de cada variable (bosque aleatorio):")
for v, imp in sorted(zip(variables, bosque.feature_importances_), key=lambda x: -x[1]):
    print(f"  {v:10} {round(imp*100, 1)}%")

Umbral aprendido (solo centros) -> acierto: 87.4% (+/- 4.6)


Regresion logistica (5 vars)    -> acierto: 84.2% (+/- 5.6)


Bosque aleatorio (5 vars)       -> acierto: 85.8% (+/- 4.0)



Importancia de cada variable (bosque aleatorio):
  crs_90     60.1%
  tklw_90    13.6%
  crdy_90    9.0%
  int_90     8.7%
  fls_90     8.6%


**El modelo mas simple gana**: el umbral sobre los centros al area alcanza un **87,4%**
de acierto, por encima del bosque aleatorio (85,8%) y de la regresion logistica (84,2%).
Las otras cuatro variables no aportan; anaden ruido.

Los centros al area explican el 60% de la decision en el bosque, confirmando que el
resto de variables son marginales.

### A3. Analisis de los errores del modelo

In [10]:
# Calculamos la probabilidad de cada jugador sin que el modelo lo haya visto
probs = cross_val_predict(tocon, defensas[["crs_90"]], y, cv=cv, method="predict_proba")

# Guardamos la probabilidad de ser central
defensas["prob_central"] = probs[:, 1]

# Generamos la prediccion a partir de la probabilidad
defensas["prediccion"] = (defensas["prob_central"] >= 0.5).astype(int)

# Calculamos el acierto real
print("Acierto real:", round((defensas["prediccion"] == defensas["es_central"]).mean()*100, 1), "%")

# Mostramos los jugadores que el modelo clasifica mal
fallos = defensas[defensas["prediccion"] != defensas["es_central"]]
print(f"\nFALLOS REALES ({len(fallos)} jugadores)")
print(fallos[["player", "team", "posicion", "crs_90"]].round(2).sort_values("crs_90").to_string(index=False))

# Entrenamos el modelo final con todos los datos para aplicarlo a Segunda
tocon.fit(defensas[["crs_90"]], y)

# Extraemos el umbral que ha aprendido el arbol
umbral_aprendido = tocon.tree_.threshold[0]
print(f"\nUmbral aprendido: {round(umbral_aprendido, 3)} centros por 90 minutos")

Acierto real: 87.4 %

FALLOS REALES (16 jugadores)
             player            team posicion  crs_90
         Juan Foyth      Villarreal  lateral    0.00
     Javi Rodríguez      Celta Vigo  lateral    0.04
        Daley Blind          Girona  lateral    0.45
        Eric García       Barcelona  central    0.46
    Nahuel Tenaglia          Alavés  lateral    0.47
      Ronald Araújo       Barcelona  central    0.50
        Marc Pubill Atlético Madrid  lateral    0.52
    Leandro Cabrera        Espanyol  central    0.54
     Valentín Gómez      Real Betis  central    0.57
       Dávid Hancko Atlético Madrid  central    1.07
        Pedro Bigas           Elche  central    1.14
Adrián de la Fuente         Levante  central    1.33
       Víctor Chust           Elche  central    1.49
       Jules Koundé       Barcelona  central    1.59
     Arnau Martinez          Girona  central    1.69
  Alejandro Francés          Girona  central    1.94

Umbral aprendido: 0.585 centros por 90 minutos


**Los 16 fallos tienen tres origenes distintos:**

1. **Siete son etiquetas desfasadas de Marca** (Foyth, Javi Rodriguez, Blind, Tenaglia,
   Pubill, Azpilicueta, Petrot): laterales de ficha que jugaron de central. WhoScored
   coincide con el modelo, no con Marca.
2. **Cinco son errores reales**: centrales confirmados por ambas fuentes que centran
   por encima del umbral (Hancko, Bigas, De la Fuente, Chust, Frances).
3. **El resto son casos donde las dos fuentes discrepan entre si** (Mourino, Unai Nunez).

Validado contra las posiciones de WhoScored en lugar de Marca, el acierto es del 89,6%,
practicamente igual: ambas fuentes son igual de dificiles de predecir.

**Conclusion honesta:** el metodo sirve como criba previa, no como clasificacion
definitiva. Los candidatos finales se verifican manualmente.

### A4. Aplicacion a Segunda Division

In [11]:
# Cargamos las estadisticas de Segunda ya descargadas
fb2 = pd.read_csv("data/fbref_misc_segunda_2526.csv", header=[0, 1], index_col=[0, 1, 2, 3])

# Aplanamos el MultiIndex de columnas
fb2.columns = [c[0] if (not c[1] or str(c[1]).startswith("Unnamed")) else c[1] for c in fb2.columns]

# Reseteamos el indice para tener equipo y jugador como columnas
fb2 = fb2.reset_index()

# Nos quedamos con los defensas con muestra suficiente
def2 = fb2[(fb2["pos"].str.startswith("DF", na=False)) & (fb2["90s"] >= 10)].copy()

# Calculamos las metricas por 90 minutos
for m in ["Crs", "Int", "TklW", "Fls", "Fld"]:
    def2[m + "_90"] = def2[m] / def2["90s"]

# Renombramos para mantener la nomenclatura del modelo
def2["crs_90"] = def2["Crs_90"]
def2["int_90"] = def2["Int_90"]
def2["tklw_90"] = def2["TklW_90"]

# Aplicamos el umbral aprendido para clasificar central o lateral
def2["posicion_pred"] = (def2["crs_90"] < umbral_aprendido).map({True: "central", False: "lateral"})

# Guardamos la tabla de defensas de Segunda clasificados
def2.to_csv("data/defensas_segunda_clasificados.csv", index=False)

# Mostramos cuantos defensas hay de cada tipo
print("Defensas de Segunda clasificados:")
print(def2["posicion_pred"].value_counts())

# Comparamos las medianas de centrales entre las dos categorias
print("\nMEDIANA DE CENTRALES")
cb2 = def2[def2["posicion_pred"] == "central"]
cb1 = defensas[defensas["posicion"] == "central"]
print("  Segunda -> centros:", round(cb2["crs_90"].median(), 2),
      "| intercepciones:", round(cb2["int_90"].median(), 2),
      "| entradas:", round(cb2["tklw_90"].median(), 2))
print("  Primera -> centros:", round(cb1["crs_90"].median(), 2),
      "| intercepciones:", round(cb1["int_90"].median(), 2),
      "| entradas:", round(cb1["tklw_90"].median(), 2))

Defensas de Segunda clasificados:
posicion_pred
lateral    73
central    64
Name: count, dtype: int64

MEDIANA DE CENTRALES
  Segunda -> centros: 0.11 | intercepciones: 1.17 | entradas: 0.8
  Primera -> centros: 0.1 | intercepciones: 1.1 | entradas: 0.92


Las distribuciones son muy similares entre categorias (centros 0,11 frente a 0,10;
intercepciones 1,17 frente a 1,10).

**Advertencia metodologica esencial:** que las medianas coincidan significa que se hace
la misma *cantidad* de acciones, no que se hagan contra rivales del mismo nivel.
Interceptar en Segunda es mas facil que en Primera porque el rival ejecuta peor.
**No se ajusta por nivel competitivo**, y no hay forma de hacerlo con estos datos.
Los candidatos deben interpretarse como perfiles compatibles con la carencia detectada,
no como equivalentes directos a jugadores de Primera.

### A5. Criba de candidatos

In [12]:
# Definimos el umbral de anticipacion un 20 por ciento sobre la mediana de Segunda
umbral_int = cb2["int_90"].median() * 1.2

# Copiamos la tabla de centrales de Segunda
cb2 = cb2.copy()

# Calculamos el percentil de cada central dentro de su categoria
cb2["pct_int"] = cb2["int_90"].rank(pct=True) * 100
cb2["pct_tklw"] = cb2["tklw_90"].rank(pct=True) * 100

# Filtramos a los que superan el umbral de anticipacion
candidatos = cb2[cb2["int_90"] >= umbral_int].sort_values("int_90", ascending=False)

# Mostramos el umbral utilizado
print("Umbral de intercepciones:", round(umbral_int, 2), "por 90 minutos")

# Mostramos los candidatos con su edad y minutos
print("\nCENTRALES DE SEGUNDA CON MAS ANTICIPACION")
print(candidatos[["player", "team", "age", "90s", "int_90", "pct_int",
                  "tklw_90", "pct_tklw", "crs_90"]].round(2).to_string(index=False))

Umbral de intercepciones: 1.41 por 90 minutos

CENTRALES DE SEGUNDA CON MAS ANTICIPACION
          player            team  age  90s  int_90  pct_int  tklw_90  pct_tklw  crs_90
    Eric Curbelo  Sporting Gijón 31.0 15.3    2.61   100.00     2.22    100.00    0.33
     Xavi Sintes         Córdoba 23.0 23.1    2.08    98.44     1.30     93.75    0.00
     Jorge Sáenz         Leganés 28.0 11.5    2.00    96.88     0.78     46.88    0.09
   Kazunari Kita Real Sociedad B 19.0 26.5    1.89    95.31     0.98     79.69    0.00
  Jawad El Yamiq        Zaragoza 33.0 11.5    1.74    93.75     0.96     75.78    0.09
          Barcia      Las Palmas 24.0 33.5    1.67    92.19     0.75     40.62    0.03
 Alberto Jiménez       Castellón 32.0 34.7    1.67    90.62     1.41     96.88    0.32
   Marc Bombardó      FC Andorra 19.0 14.7    1.63    89.06     0.88     70.31    0.14
    Nélson Monte         Almería 30.0 27.4    1.53    87.50     0.26      3.12    0.15
    Jorge Pulido          Huesca 34.0 35.

**Criterios de descarte aplicados** (criterio deportivo, no estadistico):

- **Veteranos (32+ anos)**: El Yamiq (33), Jorge Pulido (34), Carlos Hernandez (34),
  Alberto Jimenez (32). Taparian el agujero a corto plazo pero no son solucion de futuro
  para un club recien descendido.
- **Muestra insuficiente**: Eric Curbelo lidera la tabla (2,61) pero con solo 15,3
  partidos completos.
- **Filiales**: Kazunari Kita (Real Sociedad B) compite contra rivales distintos y
  pertenece a otro club.
- **Perfil defensivo incompleto**: Nelson Monte (percentil 3 en entradas) y Federico
  Bonini (percentil 13).

**Lista corta resultante:** Xavi Sintes (Cordoba, 23 anos), Mika Marmol y Sergio Barcia
(Las Palmas, 24 y 25), Federico Bonini (Almeria, 23).

### A6. Verificacion manual y valores de mercado

La clasificacion de posicion procede de un modelo con 87,4% de acierto, asi que los
finalistas se verifican uno a uno. **Los cuatro se confirmaron como defensas centrales
en Transfermarkt: 4 de 4.**

| Jugador | Equipo 2025/26 | Edad | Valor de mercado | Club actual |
|---|---|---|---|---|
| **Xavi Sintes** | Cordoba CF | 24 | **400 mil EUR** | GD Estoril Praia |
| Sergio Barcia | Las Palmas | 25 | 1,50 mill. EUR | SC Braga |
| Federico Bonini | Almeria | 23 | 3,00 mill. EUR | UD Almeria |
| Mika Marmol | Las Palmas | 24 | 10,00 mill. EUR | Feyenoord |

*Fuente: Transfermarkt.es, consultado en septiembre de 2026.*

**Limitacion importante:** los cuatro se traspasaron a equipos de mayor nivel despues de
la temporada analizada, por lo que estos valores son **posteriores** al momento en que el
Mallorca habria tenido que fichar. Marmol no valia 10 millones jugando en Las Palmas:
ese precio es consecuencia de su traspaso al Feyenoord. El historico de Transfermarkt
permite consultar el valor en la fecha relevante (Sintes figuraba en 200 mil EUR en
diciembre de 2024, en el Cordoba).

### A7. Comparacion del candidato con los centrales del Mallorca

In [13]:
# Cargamos las estadisticas de FBref de Primera
fb1 = pd.read_csv("data/fbref_misc_laliga_2526.csv", header=[0, 1], index_col=[0, 1, 2, 3])

# Aplanamos el MultiIndex de columnas
fb1.columns = [c[0] if (not c[1] or str(c[1]).startswith("Unnamed")) else c[1] for c in fb1.columns]

# Reseteamos el indice y creamos las claves de cruce
fb1 = fb1.reset_index()
fb1["clave"] = fb1["player"].apply(normalizar)
fb1["clave_equipo"] = fb1["team"].apply(normalizar)

# Nos quedamos con los centrales de Primera segun la posicion consolidada
jug["clave_j"] = jug["clave"]
cb_primera = fb1.merge(jug[["clave_equipo", "clave", "posicion"]],
                       on=["clave_equipo", "clave"], how="inner")
cb_primera = cb_primera[cb_primera["posicion"] == "central"].copy()

# Definimos las metricas comunes a ambas categorias
metricas_fb = ["Int", "TklW", "Fls", "Fld"]

# Calculamos cada metrica por 90 minutos en Primera
for m in metricas_fb:
    cb_primera[m + "_90"] = cb_primera[m] / cb_primera["90s"]

# Calculamos los percentiles dentro de cada categoria por separado
for m in metricas_fb:
    cb_primera["pct_" + m] = cb_primera[m + "_90"].rank(pct=True) * 100
    cb2["pct_" + m] = cb2[m + "_90"].rank(pct=True) * 100

# Extraemos a Sintes de los centrales de Segunda
sintes = cb2[cb2["player"].str.contains("Sintes", na=False)].iloc[0]

# Extraemos a los centrales del Mallorca
mall_cb = cb_primera[cb_primera["clave_equipo"] == "mallorca"]

# Mostramos los valores por 90 minutos
print("VALORES POR 90 MINUTOS (fuente unica: FBref)")
print(f"{'Sintes (Segunda)':26}", " ".join(f"{m}={round(sintes[m+'_90'], 2):<6}" for m in metricas_fb))
for _, f in mall_cb.iterrows():
    print(f"{f['player'] + ' (Primera)':26}", " ".join(f"{m}={round(f[m+'_90'], 2):<6}" for m in metricas_fb))

# Mostramos los percentiles dentro de su propia categoria
print("\nPERCENTIL DENTRO DE SU CATEGORIA")
print(f"{'Sintes (vs Segunda)':26}", " ".join(f"{m}={round(sintes['pct_'+m]):<5}" for m in metricas_fb))
for _, f in mall_cb.iterrows():
    print(f"{f['player'] + ' (vs Primera)':26}", " ".join(f"{m}={round(f['pct_'+m]):<5}" for m in metricas_fb))

VALORES POR 90 MINUTOS (fuente unica: FBref)
Sintes (Segunda)           Int=2.08   TklW=1.3    Fls=1.04   Fld=0.74  
Antonio Raillo (Primera)   Int=0.52   TklW=0.57   Fls=1.0    Fld=0.65  
Martin Valjent (Primera)   Int=0.65   TklW=0.9    Fls=0.98   Fld=0.33  

PERCENTIL DENTRO DE SU CATEGORIA
Sintes (vs Segunda)        Int=98    TklW=94    Fls=39    Fld=59   
Antonio Raillo (vs Primera) Int=7     TklW=10    Fls=42    Fld=58   
Martin Valjent (vs Primera) Int=18    TklW=43    Fls=38    Fld=15   


### Ficha del candidato: Xavi Sintes

| | |
|---|---|
| **Posicion** | Defensa central *(verificado en Transfermarkt)* |
| **Edad** | 24 anos |
| **Equipo 2025/26** | Cordoba CF (Segunda Division) |
| **Muestra** | 23,1 partidos completos |
| **Intercepciones** | 2,08 por 90 min (percentil 98 de Segunda) |
| **Entradas ganadas** | 1,30 por 90 min (percentil 94) |
| **Valor de mercado** | 400 mil EUR |

### Por que encaja

1. El Mallorca concedia el 91% de su exceso de xG en **juego abierto** y por el
   **pasillo central**, no a balon parado.
2. **Ninguno de sus cuatro centrales** alcanzaba la mediana de LaLiga en intercepciones
   (0,86 / 0,70 / 0,62 / 0,41 frente a 1,06): no habia recambio interno.
3. Sintes intercepta 2,08 por 90, frente a 0,65 de Valjent y 0,52 de Raillo.
4. Lo consigue **sin cometer mas faltas** que la media (1,04 frente a 1,00 y 0,98, los
   tres en torno al percentil 40): llega antes al balon, no es mas agresivo.
5. Su valor de mercado lo hace **asumible para un club recien descendido**.

### Limitaciones a declarar

- Los datos de Segunda proceden solo de FBref: **no hay despejes, bloqueos ni duelos
  aereos**, por lo que no puede evaluarse su juego aereo ni su solvencia en el area.
- **No se ajusta por nivel competitivo** entre categorias: los percentiles de Sintes son
  frente a Segunda y los de Valjent y Raillo frente a Primera.
- La posicion se clasifico con un modelo (87,4% de acierto) y se verifico manualmente.
- El valor de mercado disponible es posterior al periodo analizado.